# Step 3 — Aggregation Pipelines: `$project`, `$out`, and `$lookup`

In this activity we'll work through two guided examples that combine multiple aggregation stages.
By the end you should be comfortable with:

| Stage | What it does |
|---|---|
| `$project` | Reshape documents — rename, compute, include/exclude fields |
| `$unwind` | Flatten an array field into one doc per element |
| `$lookup` | Left-outer-join another collection |
| `$out` | Write pipeline results to a **new collection** |

These are exactly the tools you'll need for the Step 5 COVID homework.

---
## Setup

In [ ]:
!pip install pymongo

In [ ]:
import pymongo

# Replace with your connection string from Step 4
uri = "invalid_url"  ## PUT YOUR CONNECTION STRING HERE

client = pymongo.MongoClient(uri)
client.admin.command('ping')
print("Connected!")

In [ ]:
# Download the data files

!curl -sL "https://raw.githubusercontent.com/byu-cs-452/byu-cs-452-class-content/refs/heads/main/data/movies.json"     > movies.json
!curl -sL "https://raw.githubusercontent.com/byu-cs-452/byu-cs-452-class-content/refs/heads/main/data/ratings.json"    > ratings.json
!curl -sL "https://raw.githubusercontent.com/byu-cs-452/byu-cs-452-class-content/refs/heads/main/data/box_office.json" > box_office.json

print("Downloaded!")

In [ ]:
import json

db = client["movies"]

def load_collection(name):
    col = db[name]
    if col.count_documents({}) == 0:
        with open(f"{name}.json") as f:
            col.insert_many(json.load(f))
        print(f"Loaded {col.count_documents({})} docs into '{name}'")
    else:
        print(f"'{name}' already has {col.count_documents({})} docs")

load_collection("movies")
load_collection("ratings")
load_collection("box_office")

---
## Example 1 — `$project` + `$group`: Revenue per year

**Goal:** For each release year, compute the total and average worldwide gross.

This shows how `$project` lets you reshape documents *before* grouping — 
here we extract just the fields we care about and rename them for clarity.

In [ ]:
pipeline = [
    # Step 1: Lookup box_office data onto each movie
    {"$lookup": {
        "from": "box_office",
        "localField": "title",
        "foreignField": "title",
        "as": "box"
    }},
    # Step 2: Unwind the joined array (1 box_office doc per movie)
    {"$unwind": "$box"},
    # Step 3: Project only the fields we need, with clean names
    {"$project": {
        "_id": 0,
        "title": 1,
        "year": 1,
        "worldwide_gross": "$box.worldwide_gross"
    }},
    # Step 4: Group by year
    {"$group": {
        "_id": "$year",
        "total_gross": {"$sum": "$worldwide_gross"},
        "avg_gross":   {"$avg": "$worldwide_gross"},
        "movie_count": {"$sum": 1}
    }},
    {"$sort": {"_id": 1}}
]

for doc in db.movies.aggregate(pipeline):
    print(doc)

### Key takeaways
- `$project` acts like SQL `SELECT col AS alias` — it reshapes every document.
- You can reference nested/joined fields with dot notation like `"$box.worldwide_gross"`.
- Stages execute **in order** — a pipeline is just a list of transformations.

---
## Example 2 — `$out`: Save results as a new collection

**Goal:** Create a flattened `movie_summary` collection that joins movies with their ratings,
keeping only `title`, `genre`, `country`, and `rating`.

`$out` writes the pipeline output to a collection — think of it like `CREATE TABLE AS SELECT` in SQL.
This is useful for materializing views or preparing data for downstream queries.

In [ ]:
pipeline = [
    # Join ratings onto movies
    {"$lookup": {
        "from": "ratings",
        "localField": "title",
        "foreignField": "title",
        "as": "rating_info"
    }},
    {"$unwind": {"path": "$rating_info", "preserveNullAndEmptyArrays": True}},
    # Keep only the fields we want
    {"$project": {
        "_id": 0,
        "title": 1,
        "genre": 1,
        "country": 1,
        "rating": {"$ifNull": ["$rating_info.rating", "N/A"]}
    }},
    # Write to a new collection
    {"$out": "movie_summary"}
]

# Run the pipeline (output goes to the collection, not the cursor)
db.movies.aggregate(pipeline)

# Verify
print(f"movie_summary now has {db.movie_summary.count_documents({})} documents")
for doc in db.movie_summary.find().limit(5):
    print(doc)

### Key takeaways
- `$out` **replaces** the target collection if it already exists — be careful!
- `$out` must be the **last stage** in the pipeline.
- `preserveNullAndEmptyArrays: True` keeps movies even if they have no matching rating (like a LEFT JOIN).

---
## Now you try!

Use the same `movies`, `ratings`, and `box_office` collections for the exercises below.

### Exercise 1

Find the **top-rated movie per genre**.

Hints:
- `$lookup` ratings onto movies
- `$project` to keep `title`, `genre`, and `rating`
- `$sort` by rating descending, then `$group` with `$first`

In [ ]:
# your code here

### Exercise 2

Create a new collection called `genre_report` (using `$out`) that contains one document per genre with:
- `genre` — the genre name
- `movie_count` — number of movies  
- `avg_worldwide_gross` — average box office worldwide gross  
- `avg_rating` — average rating

Hints:
- You'll need **two** `$lookup` stages (one for ratings, one for box_office)
- `$unwind` each joined array
- `$group` by genre, then `$project` to clean up the field names
- End with `$out`

In [ ]:
# your code here

In [ ]:
# Verify your genre_report collection
for doc in db.genre_report.find().sort("avg_worldwide_gross", -1):
    print(doc)